# Chess Personas — Colab starter
Train Hikaru- and Caruana-inspired adapters on a frozen Maia-3 backbone.

Select **Runtime → Change runtime type → GPU**. This notebook starts with three
months of public games and a position cap. It does not launch automatically.
Caruana remains a provisional second persona until clock coverage is checked.

For a private repository, download its ZIP from GitHub's **Code → Download ZIP**
menu, then upload it below. Credentials are not embedded in this notebook.

In [ ]:
from pathlib import Path
import io, os, subprocess, sys, zipfile
from google.colab import files

workspace = Path('/content/chess-personas-work')
workspace.mkdir(exist_ok=True)
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one repository ZIP'
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(workspace)
projects = [p.parent for p in workspace.rglob('pyproject.toml')
            if (p.parent / 'requirements-maia.txt').exists()]
assert len(projects) == 1, 'Expected one Chess Personas repository'
os.chdir(projects[0])
print('Repository:', Path.cwd())

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)

run('-m', 'pip', 'install', '-e', '.[train,dev]')
run('-m', 'pip', 'install', '-r', 'requirements-maia.txt')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch to GPU for training')
run('-m', 'pytest', '-q')

## Download and inspect data
Public archive requests are sequential and cached. Increase the month count only
after the first run works. A full-history request is `--max-months 0`.
If preparing a second time, choose a new output directory to preserve old splits.

In [ ]:
run('-m', 'chess_personas.cli', 'download', 'hikaru', 'fabianocaruana', '--max-months', '3')
run('-m', 'chess_personas.cli', 'prepare', 'data/raw/*/*.json', '--output', 'data/processed')
import json
audit = json.loads(Path('data/processed/audit.json').read_text())
print(json.dumps(audit, indent=2))

## Train
Check that both players have training and validation positions and useful timing
coverage. The default cap is 10,000 positions per player per split. A batch size of
32 is a starting point; use 8 if you run out of GPU memory. Maia weights download
on the first run. No prediction accuracy or Elo is implied by this starter.

In [ ]:
run('-m', 'chess_personas.cli', 'train', '--output', 'runs/first',
    '--epochs', '3', '--batch-size', '32', '--max-per-player', '10000')

## Held-out evaluation and example prediction
Compare adapted and base move metrics, plus timing NLL and interval coverage.
The example observation is synthetic; it is not a recorded Hikaru game.

In [ ]:
run('-m', 'chess_personas.cli', 'evaluate', 'runs/first/best.pt',
    '--output', 'runs/first/test-metrics.json')
run('-m', 'chess_personas.cli', 'predict', 'runs/first/best.pt',
    '--record', 'examples/position.json')

## Save your run before the session ends
The checkpoint includes the frozen backbone plus trained adapters and timing head.
This initial trainer starts fresh for each run; it does not resume optimizer state.

In [ ]:
import shutil
archive_path = shutil.make_archive('/content/chess-personas-first-run', 'zip', 'runs/first')
files.download(archive_path)